# 04 — Silver Validation
## Formal Data Quality Gate Before Gold

**Purpose:** Run structured validation checks on the enriched Silver table.  
CRITICAL failures halt the pipeline. WARNINGS are logged and monitored.  
This notebook is the quality contract between Silver and Gold layers.

**Input :** `sales_silver.clean_superstore`  
**Output:** Validation report (printed + logged). Pipeline halts on CRITICAL failure.  
**Layer  :** Silver (validate)


## 1. Load config

In [0]:
%run ./00_setup_and_config

In [0]:
log("INFO", "04_silver_validate", "Starting Silver validation gate")


## 2. Read Silver table and setup results collector

In [0]:
# Every check appends a result dict to `validation_results`.
# At the end we evaluate the full list — not one check at a time.
# This means the pipeline runs ALL checks even if the first one fails,
# giving you a complete picture of data quality rather than stopping at the first problem.

df = spark.table(SILVER_TABLE)
total_rows = df.count()

# Results collector — every check appends here
validation_results = []

def add_result(rule, status, actual, expected, severity="CRITICAL", note=""):
    """
    Append a validation result to the collector.

    Args:
        rule     : Human-readable rule name
        status   : "PASS" or "FAIL"
        actual   : What the data actually contains
        expected : What we expected
        severity : "CRITICAL" (halts pipeline) or "WARNING" (logged only)
        note     : Optional explanation
    """
    validation_results.append({
        "rule"     : rule,
        "status"   : status,
        "actual"   : str(actual),
        "expected" : str(expected),
        "severity" : severity,
        "note"     : note,
    })
    icon = "✅" if status == "PASS" else ("🔴" if severity == "CRITICAL" else "🟡")
    print(f"  {icon} [{severity:<8}] {rule:<45} | actual: {actual}")


log("INFO", "04_silver_validate", f"Silver table loaded — {total_rows:,} rows")
print(f"  Rows to validate : {total_rows:,}")
print(f"  Columns          : {len(df.columns)}")
print()


## 3. Row count checks

In [0]:
# Rule 1: Minimum row count
#   The Superstore dataset has 9,994 rows. After cleaning we should
#   retain at least 99% — losing more than 1% is suspicious and
#   warrants investigation before proceeding to Gold.
#
# Rule 2: Maximum row count
#   Silver should never have MORE rows than Bronze (no duplicates added).
#   If Silver > Bronze, something went wrong in the write logic.

print("=== CHECK 1: ROW COUNT SANITY ===\n")

bronze_df    = spark.table(BRONZE_TABLE)
bronze_rows  = bronze_df.count()
min_expected = int(bronze_rows * 0.99)   # 99% retention threshold
max_expected = bronze_rows               # Silver must not exceed Bronze

# Rule 1: Minimum retention
add_result(
    rule     = "Row count >= 99% of Bronze",
    status   = "PASS" if total_rows >= min_expected else "FAIL",
    actual   = f"{total_rows:,}",
    expected = f">= {min_expected:,}",
    severity = "CRITICAL",
    note     = f"Retention: {total_rows/bronze_rows*100:.2f}%"
)

# Rule 2: No row inflation
add_result(
    rule     = "Silver rows <= Bronze rows",
    status   = "PASS" if total_rows <= max_expected else "FAIL",
    actual   = f"{total_rows:,}",
    expected = f"<= {max_expected:,}",
    severity = "CRITICAL",
    note     = "Inflation means duplicate rows were introduced"
)


## 4. Schema checks — correct columns and correct types

In [0]:
# If Silver has wrong types (e.g. order_date became string again),
# Gold table creation will silently produce wrong results.

from pyspark.sql.types import (
    StringType, IntegerType, DoubleType,
    DateType, BooleanType, LongType
)

print("\n=== CHECK 2: SCHEMA VALIDATION ===\n")

# Expected column → type mapping for critical columns
expected_schema = {
    "order_id"        : StringType(),
    "order_date"      : DateType(),
    "ship_date"       : DateType(),
    "customer_id"     : StringType(),
    "customer_name"   : StringType(),
    "segment"         : StringType(),
    "region"          : StringType(),
    "category"        : StringType(),
    "sub_category"    : StringType(),
    "product_id"      : StringType(),
    "product_name"    : StringType(),
    "sales"           : DoubleType(),
    "quantity"        : IntegerType(),
    "discount"        : DoubleType(),
    "profit"          : DoubleType(),
    "postal_code"     : StringType(),
    "shipping_days"   : IntegerType(),
    "profit_margin"   : DoubleType(),
    "is_profitable"   : BooleanType(),
    "order_year"      : IntegerType(),
    "order_month"     : IntegerType(),
    "order_quarter"   : IntegerType(),
}

actual_schema = {f.name: f.dataType for f in df.schema.fields}

for col_name, expected_type in expected_schema.items():

    # Check column exists
    if col_name not in actual_schema:
        add_result(
            rule     = f"Column exists: {col_name}",
            status   = "FAIL",
            actual   = "MISSING",
            expected = expected_type.simpleString(),
            severity = "CRITICAL",
            note     = "Column not found in Silver schema"
        )
        continue

    # Check column type
    actual_type = actual_schema[col_name]
    type_match  = type(actual_type) == type(expected_type)

    add_result(
        rule     = f"Type correct: {col_name}",
        status   = "PASS" if type_match else "FAIL",
        actual   = actual_type.simpleString(),
        expected = expected_type.simpleString(),
        severity = "CRITICAL",
        note     = "" if type_match else "Type mismatch — check casting in Notebook 02"
    )


## 5. Null completeness checks

In [0]:
# After Silver cleaning, specific columns must have zero nulls.
# These are columns that Gold tables join on — a null foreign key breaks the entire star schema.

from pyspark.sql import functions as F

print("\n=== CHECK 3: NULL COMPLETENESS ===\n")

# Columns that must be 100% non-null for Gold to work correctly
zero_null_cols = [
    "order_id", "order_date", "ship_date",
    "customer_id", "customer_name",
    "product_id", "product_name",
    "region", "category", "segment",
    "sales", "quantity", "profit",
    "shipping_days", "profit_margin",
    "order_year", "order_month",
]

null_counts = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in zero_null_cols
    if c in df.columns
]).collect()[0]

for col_name, null_count in zip(zero_null_cols, null_counts):
    if col_name not in df.columns:
        continue
    add_result(
        rule     = f"Zero nulls: {col_name}",
        status   = "PASS" if null_count == 0 else "FAIL",
        actual   = f"{null_count} nulls",
        expected = "0 nulls",
        severity = "CRITICAL",
        note     = "Null in join key breaks Gold star schema" if null_count > 0 else ""
    )


## 6. Business rule checks

In [0]:
# These rules encode domain knowledge about what valid sales data looks like.
# Some are CRITICAL (wrong values corrupt Gold aggregations).
# Some are WARNING (suspicious but pipeline can continue).

print("\n=== CHECK 4: BUSINESS RULES ===\n")

# Rule: sales must be positive
sales_zero_or_neg = df.filter(F.col("sales") <= 0).count()
add_result(
    rule     = "All sales > 0",
    status   = "PASS" if sales_zero_or_neg == 0 else "FAIL",
    actual   = f"{sales_zero_or_neg} violations",
    expected = "0 violations",
    severity = "CRITICAL"
)

# Rule: quantity must be >= 1
qty_invalid = df.filter(F.col("quantity") < 1).count()
add_result(
    rule     = "All quantity >= 1",
    status   = "PASS" if qty_invalid == 0 else "FAIL",
    actual   = f"{qty_invalid} violations",
    expected = "0 violations",
    severity = "CRITICAL"
)

# Rule: discount between 0 and 1
discount_invalid = df.filter(
    (F.col("discount") < 0) | (F.col("discount") > 1)
).count()
add_result(
    rule     = "Discount between 0 and 1",
    status   = "PASS" if discount_invalid == 0 else "FAIL",
    actual   = f"{discount_invalid} violations",
    expected = "0 violations",
    severity = "CRITICAL"
)

# Rule: ship_date must be >= order_date
ship_before_order = df.filter(
    F.col("ship_date") < F.col("order_date")
).count()
add_result(
    rule     = "ship_date >= order_date",
    status   = "PASS" if ship_before_order == 0 else "FAIL",
    actual   = f"{ship_before_order} violations",
    expected = "0 violations",
    severity = "CRITICAL"
)

# Rule: shipping_days should be reasonable (0-30 days)
shipping_outliers = df.filter(
    (F.col("shipping_days") < 0) | (F.col("shipping_days") > 30)
).count()
add_result(
    rule     = "Shipping days between 0 and 30",
    status   = "PASS" if shipping_outliers == 0 else "FAIL",
    actual   = f"{shipping_outliers} violations",
    expected = "0 violations",
    severity = "WARNING",
    note     = "Outliers flagged but pipeline continues"
)

# Rule: profit_margin should be between -1 and 1
# (margins outside this range suggest data corruption)
margin_outliers = df.filter(
    (F.col("profit_margin") < -1) | (F.col("profit_margin") > 1)
).count()
add_result(
    rule     = "Profit margin between -1 and 1",
    status   = "PASS" if margin_outliers == 0 else "FAIL",
    actual   = f"{margin_outliers} violations",
    expected = "0 violations",
    severity = "WARNING",
    note     = "Extreme margins may indicate pricing errors"
)


## 7. Categorical domain checks

In [0]:
# Known valid values for key categorical columns.
# An unexpected value (e.g. region = "Nrth" instead of "North"), means a standardisation step failed and would silently create a new region in dim_region that analysts won't recognise.

print("\n=== CHECK 5: CATEGORICAL DOMAINS ===\n")

valid_domains = {
    "segment"  : {"Consumer", "Corporate", "Home Office"},
    "region"   : {"East", "West", "Central", "South"},
    "category" : {"Furniture", "Office Supplies", "Technology"},
}

for col_name, valid_values in valid_domains.items():
    if col_name not in df.columns:
        continue

    # Find values in the data not in the valid set
    actual_values = {
        row[col_name]
        for row in df.select(col_name).distinct().collect()
    }
    unexpected = actual_values - valid_values

    add_result(
        rule     = f"Valid domain: {col_name}",
        status   = "PASS" if not unexpected else "FAIL",
        actual   = str(actual_values),
        expected = str(valid_values),
        severity = "CRITICAL",
        note     = f"Unexpected values: {unexpected}" if unexpected else ""
    )


## 8. Consistency checks between related columns

In [0]:
# Cell 8: Consistency checks between related columns
#
# These checks verify that derived columns are consistent with
# the source columns they were derived from.
# If profit_margin = 0.25 but profit/sales = 0.30, something
# went wrong in the transformation step.

print("\n=== CHECK 6: DERIVED COLUMN CONSISTENCY ===\n")

# Check: order_year matches year extracted from order_date
year_mismatch = df.filter(
    F.col("order_year") != F.year("order_date")
).count()
add_result(
    rule     = "order_year matches year(order_date)",
    status   = "PASS" if year_mismatch == 0 else "FAIL",
    actual   = f"{year_mismatch} mismatches",
    expected = "0 mismatches",
    severity = "CRITICAL",
    note     = "order_year derivation may be incorrect"
)

# Check: shipping_days matches datediff(ship_date, order_date)
shipping_mismatch = df.filter(
    F.col("shipping_days") != F.datediff("ship_date", "order_date")
).count()
add_result(
    rule     = "shipping_days matches datediff",
    status   = "PASS" if shipping_mismatch == 0 else "FAIL",
    actual   = f"{shipping_mismatch} mismatches",
    expected = "0 mismatches",
    severity = "CRITICAL",
    note     = "shipping_days derivation may be incorrect"
)

# Check: is_profitable consistent with profit sign
profitability_mismatch = df.filter(
    ((F.col("profit") > 0)  & (~F.col("is_profitable"))) |
    ((F.col("profit") <= 0) & ( F.col("is_profitable")))
).count()
add_result(
    rule     = "is_profitable consistent with profit",
    status   = "PASS" if profitability_mismatch == 0 else "FAIL",
    actual   = f"{profitability_mismatch} mismatches",
    expected = "0 mismatches",
    severity = "CRITICAL"
)


## 9. Generate the full validation report

In [0]:
from datetime import datetime

print("\n")
print("=" * 65)
print("  DATA QUALITY VALIDATION REPORT")
print(f"  Table    : {SILVER_TABLE}")
print(f"  Run time : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  Rows     : {total_rows:,}")
print("=" * 65)

total_checks   = len(validation_results)
passed_checks  = sum(1 for r in validation_results if r["status"] == "PASS")
failed_checks  = sum(1 for r in validation_results if r["status"] == "FAIL")
critical_fails = sum(
    1 for r in validation_results
    if r["status"] == "FAIL" and r["severity"] == "CRITICAL"
)
warning_fails  = sum(
    1 for r in validation_results
    if r["status"] == "FAIL" and r["severity"] == "WARNING"
)

print(f"\n  Total checks    : {total_checks}")
print(f"  Passed          : {passed_checks}  ✅")
print(f"  Failed          : {failed_checks}  {'🔴' if critical_fails > 0 else '🟡'}")
print(f"    Critical fails: {critical_fails}  🔴")
print(f"    Warnings      : {warning_fails}  🟡")
print(f"\n  Pass rate       : {passed_checks/total_checks*100:.1f}%")
print("=" * 65)

# Detail on failures only
failures = [r for r in validation_results if r["status"] == "FAIL"]
if failures:
    print("\n  FAILED CHECKS:\n")
    for r in failures:
        icon = "🔴" if r["severity"] == "CRITICAL" else "🟡"
        print(f"  {icon} [{r['severity']:<8}] {r['rule']}")
        print(f"       Actual   : {r['actual']}")
        print(f"       Expected : {r['expected']}")
        if r["note"]:
            print(f"       Note     : {r['note']}")
        print()


## 10. Halt on CRITICAL failures

In [0]:
# This is the most important cell in the notebook.
# If any CRITICAL check failed, we raise an exception.
# This stops the pipeline from writing bad data to Gold.
#
# WARNING failures are logged but do not stop the pipeline.
# The engineer is notified but Gold proceeds.

print("=" * 65)
print("  PIPELINE GATE DECISION")
print("=" * 65)

if critical_fails > 0:
    log("ERROR", "04_silver_validate",
        f"PIPELINE HALTED — {critical_fails} CRITICAL validation failures")
    print(f"\n  🔴 PIPELINE HALTED")
    print(f"     {critical_fails} CRITICAL failure(s) detected.")
    print(f"     Fix the issues above and re-run from Notebook 02.")
    print(f"     Gold layer will NOT be written until all CRITICAL checks pass.")
    raise Exception(
        f"Silver validation failed: {critical_fails} CRITICAL checks failed. "
        f"See validation report above."
    )

elif warning_fails > 0:
    log("WARN", "04_silver_validate",
        f"Validation passed with {warning_fails} warning(s) — review recommended")
    print(f"\n  🟡 PIPELINE CONTINUES WITH WARNINGS")
    print(f"     {warning_fails} warning(s) logged — review before production use.")
    print(f"     Gold layer will proceed.")

else:
    log("INFO", "04_silver_validate",
        f"All {total_checks} checks passed — Silver is Gold-ready")
    print(f"\n  ✅ ALL CHECKS PASSED")
    print(f"     Silver data is validated and Gold-ready.")
    print(f"     Proceed to Notebook 05.")

print("=" * 65)


## 11. Persist the validation results as a Delta table

In [0]:
# In production, validation results are stored so you can:
#   1. Track quality trends over time (is data getting better or worse?)
#   2. Query historical run results in a dashboard
#   3. Alert when pass rate drops below a threshold
#
# This is the foundation of a DataObs (Data Observability) layer.

from pyspark.sql.types import StructType, StructField, StringType

schema = StructType([
    StructField("rule",     StringType(), True),
    StructField("status",   StringType(), True),
    StructField("actual",   StringType(), True),
    StructField("expected", StringType(), True),
    StructField("severity", StringType(), True),
    StructField("note",     StringType(), True),
])

df_results = spark.createDataFrame(validation_results, schema=schema)

df_results = df_results.withColumns({
    "_validated_at"    : F.current_timestamp(),
    "_validated_table" : F.lit(SILVER_TABLE),
    "_pass_rate"       : F.lit(f"{passed_checks/total_checks*100:.1f}%"),
})

(
    df_results
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER_DB}.validation_results")
)

print(f"  ✅ Validation results saved to {SILVER_DB}.validation_results")
log("INFO", "04_silver_validate", "Validation results persisted to Delta")


## 12. Notebook completion summary

In [0]:
# Cell 13: Notebook completion

print("=" * 55)
print("  SILVER VALIDATION — COMPLETE")
print("=" * 55)
print(f"  Checks run  : {total_checks}")
print(f"  Passed      : {passed_checks}")
print(f"  Failed      : {failed_checks}")
print(f"  Pass rate   : {passed_checks/total_checks*100:.1f}%")
print(f"  Gate result : {'✅ PASSED' if critical_fails == 0 else '🔴 HALTED'}")
print("=" * 55)

log("INFO", "04_silver_validate", "Notebook 04 complete ✅")